# Genotype–Phenotype Heterogeneity and Heterogeneity-Driven Infertility Management Exploration with `mlcroissant`
This notebook provides a template for loading and exploring the FAIR^2 dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Define the dataset URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.cbsv-djdb/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)

# Print basic metadata
metadata = dataset.metadata  # Keep as object for direct attribute access
print(f"{metadata.name}: {metadata.description}")
print(f"License: {getattr(metadata, 'license', 'N/A')}")
print(f"Keywords: {', '.join(getattr(metadata, 'keywords', []))}")


## 2. Data Overview
Review available record sets, fields, and their IDs.

We will list the schema record sets, their fields, and show a sample of their structure referencing `@id`.

In [ ]:
# List available record sets and their fields by @id
record_sets = dataset.record_sets
print("Available Record Sets:")
for rs in record_sets:
    print(f"- RecordSet @id: {rs['@id']}, name: {rs.get('name', 'N/A')}")

# For each record set, list fields by @id
for rs in record_sets:
    print(f"\nFields in RecordSet @id: {rs['@id']} (name: {rs.get('name', 'N/A')}):")
    fields = rs.get('fields', [])
    for fld in fields:
        print(f"  - Field @id: {fld['@id']}, name: {fld.get('name', 'N/A')}, dataType: {fld.get('dataType', 'N/A')}")

# Show sample records for each record set
for rs in record_sets:
    print(f"\nSample records from RecordSet @id: {rs['@id']}:")
    try:
        for i, rec in enumerate(dataset.records(record_set=rs['@id'])):
            print(f"Record {i+1}: {rec}")
            if i > 1: break
    except Exception as e:
        print(f"  Could not load records for this record set: {e}")

## 3. Data Extraction
Load data from each record set into DataFrames for analysis. Use the record set and field `@id`s from the overview.

In [ ]:
# Extract data from all record sets, store in dataframes by @id
dataframes = {}

# List of record set @id's
record_set_ids = [rs['@id'] for rs in dataset.record_sets]
print("Extracting data for record set @ids:", record_set_ids)

for record_set_id in record_set_ids:
    records = list(dataset.records(record_set=record_set_id))
    df = pd.DataFrame(records)
    dataframes[record_set_id] = df
    print(f"\nColumns for RecordSet @id {record_set_id}: {df.columns.tolist()}")
    print(df.head())

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and grouping data.

We select a record set containing numeric clinical data and perform filtering and normalization referencing columns/fields by their `@id`.

In [ ]:
# Choose a record set and numeric field for EDA
# For demonstration, select the first record set with a numeric field
selected_record_set_id = None
numeric_field_id = None
group_field_id = None

for rs in dataset.record_sets:
    df = dataframes[rs['@id']]
    # Scan for numeric fields
    for fld in rs.get('fields', []):
        dtype = fld.get('dataType', '').lower()
        if dtype in ['integer', 'float', 'number']:
            if fld['@id'] in df.columns:
                selected_record_set_id = rs['@id']
                numeric_field_id = fld['@id']
                # Assign a group field if categorical exists
                for gfld in rs.get('fields', []):
                    if gfld.get('dataType', '').lower() in ['text', 'string', 'boolean'] and gfld['@id'] in df.columns:
                        group_field_id = gfld['@id']
                        break
                break
    if selected_record_set_id and numeric_field_id:
        break

print(f"Selected record set @id: {selected_record_set_id}")
print(f"Selected numeric field @id: {numeric_field_id}")
print(f"Selected group field @id: {group_field_id}")

df = dataframes[selected_record_set_id]
# Drop NA for the numeric field
df = df.dropna(subset=[numeric_field_id])
# Ensure numeric
df[numeric_field_id] = pd.to_numeric(df[numeric_field_id], errors='coerce')

threshold = df[numeric_field_id].mean() if len(df) > 0 else 0
filtered_df = df[df[numeric_field_id] > threshold]
print(f"Filtered records with {numeric_field_id} > {threshold:.2f}:")
print(filtered_df.head())

normalized_col = f"{numeric_field_id}_normalized"
filtered_df[normalized_col] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
print(f"Normalized {numeric_field_id} for filtered records:")
print(filtered_df[[numeric_field_id, normalized_col]].head())

if group_field_id and group_field_id in df.columns:
    grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().reset_index()
    print(f"Grouped (mean {numeric_field_id}) by {group_field_id}:")
    print(grouped_df.head())

## 5. Visualization
Visualize numeric distributions and relationships (e.g. boxplot, histogram) between fields in the dataset, referencing columns by their `@id`.

In [ ]:
# Visualize the distribution of the selected numeric field
if not filtered_df.empty:
    plt.figure(figsize=(8, 4))
    sns.histplot(filtered_df[numeric_field_id], bins=10, kde=True)
    plt.title(f"Histogram of {numeric_field_id} (Filtered)")
    plt.xlabel(numeric_field_id)
    plt.show()

    # If group_field exists, show boxplot
    if group_field_id and group_field_id in filtered_df.columns:
        plt.figure(figsize=(8, 4))
        sns.boxplot(x=filtered_df[group_field_id], y=filtered_df[numeric_field_id])
        plt.title(f"Boxplot of {numeric_field_id} grouped by {group_field_id}")
        plt.xlabel(group_field_id)
        plt.ylabel(numeric_field_id)
        plt.show()

## 6. Conclusion
This notebook demonstrated how to extract FAIR^2 clinical dataset tables using the Croissant schema and reference all entities via their `@id` using the `mlcroissant` library.
- The dataset includes three female patients with non-classical 21-hydroxylase deficiency-associated infertility.
- Each record set represents clinical, hormone, imaging, or genotype data. All exploration and processing referenced entities by `@id`.
- Filtering, normalization, and group analysis of numeric fields provide insight into patient heterogeneity.

**Note:** Due to small sample size and scope, analytical depth is limited but the schema and processing pipeline scale easily to larger datasets.